# Support Vector Machines: Zero to Hero

The model that made "just add more dimensions" a practical idea instead of an expensive one.

> **Prerequisites:** [`ml_foundations_zero_to_hero.ipynb`](ml_foundations_zero_to_hero.ipynb)
> for the shared workflow, and [`logistic_regression_zero_to_hero.ipynb`](logistic_regression_zero_to_hero.ipynb)
> for the contrast — both draw a **linear decision boundary**, and comparing *how they choose
> it* is the fastest way to understand either.

---

## Why this one is worth your time

SVMs were the state of the art for roughly a decade, and were then displaced on large tabular
data by the tree ensembles in NB-04 and NB-05. So why learn them?

- **The kernel trick is one of the genuinely beautiful ideas in machine learning**, and it
  appears far beyond SVMs — Gaussian processes, kernel PCA, kernel ridge regression, and the
  modern "random features" literature all rest on it.
- **They are still the right tool in a real niche:** small-to-medium datasets, high-dimensional
  data, and especially $p > n$ problems (text, genomics, spectroscopy) where tree ensembles
  struggle and you cannot afford deep learning.
- **The margin idea is the cleanest introduction to structural risk minimisation** — the notion
  that among models that fit the data, you should prefer the one with the most "room to be
  wrong".
- It is asked about constantly in interviews, and the answers are unusually crisp.

## Contents

| Part | What it covers |
|---|---|
| **0. Setup** | Install + imports |
| **1. Theory from zero** | The maximum margin · why only support vectors matter · soft margin and `C` · hinge loss · **the kernel trick** · RBF and `gamma` · why scaling is mandatory |
| **2. Worked example** | Handwritten digits — the kind of problem SVMs are still good at |
| **3. The practical limits** | Why it does not scale, what `probability=True` really does, multiclass |
| **4. Tough questions** | 12 questions + 3 coding challenges |
| **5. Practice datasets** | 5 datasets with briefs |
| **6. Reading the literature** | The papers behind each section |
| **Appendix** | SVM-specific errors and a checklist |

## The one-paragraph summary

An SVM draws the **widest possible corridor** between two classes. Only the points that touch
the corridor's edges — the **support vectors** — determine where it sits; every other training
point could be deleted without changing the model. Real data is rarely separable, so a
parameter **C** buys tolerance for violations, trading margin width against training errors.
The clever part is that the whole optimisation depends on the data only through **inner
products** between pairs of points, so you can replace that inner product with a **kernel** —
computing what the dot product *would have been* in a vastly higher-dimensional space, without
ever going there.

---
# Part 0 - Setup

In [ ]:
# ---------------------------------------------------------------------------
# One-time setup. Only missing packages are installed, so re-running is cheap.
# ---------------------------------------------------------------------------
import importlib.util
import subprocess
import sys

REQUIRED = [
    ("numpy", "numpy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"),
    ("scipy", "scipy"), ("sklearn", "scikit-learn"),
]
missing = [pip for mod, pip in REQUIRED if importlib.util.find_spec(mod) is None]
if missing:
    print("installing:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("done")
else:
    print("all packages present")

In [ ]:
# ---------------------------------------------------------------------------
# Every import this notebook uses.
# ---------------------------------------------------------------------------
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.svm import SVC, SVR, LinearSVC, LinearSVR, OneClassSVM
from sklearn.datasets import (
    make_classification, make_circles, make_moons, make_blobs,
    load_breast_cancer, load_digits,
)
from sklearn.model_selection import (
    train_test_split, cross_val_score, cross_validate, StratifiedKFold, GridSearchCV,
)
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.kernel_approximation import Nystroem, RBFSampler
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, log_loss, brier_score_loss,
    confusion_matrix, classification_report,
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["figure.dpi"] = 110

print("ready | numpy", np.__version__, "| pandas", pd.__version__)

---
# Part 1 - Theory from zero

1. The maximum margin: which line, out of infinitely many?
2. Only the support vectors matter
3. Real data is not separable: the soft margin and `C`
4. Hinge loss — and how it differs from logistic regression's
5. **The kernel trick**
6. RBF and `gamma`: a two-dimensional bias-variance grid
7. Why scaling is not optional

## 1.1 The maximum margin

Take two linearly separable classes. **Infinitely many** straight lines separate them
perfectly — every one scores 100% on the training data. Logistic regression picks one by
maximising likelihood. An SVM picks by a different principle:

> **Choose the boundary that is as far as possible from the nearest point of either class.**

The distance from the boundary to the nearest point is the **margin**, and the SVM maximises
it. The intuition is about *robustness*: a boundary squeezed against the training points will
misclassify a new point that lands slightly differently, while a boundary sitting in the
middle of a wide corridor has room to absorb that.

Formally, for a boundary $w^\top x + b = 0$ with all points correctly classified and scaled so
the closest ones satisfy $|w^\top x + b| = 1$, the margin width is $2/\lVert w\rVert$. So
maximising the margin means **minimising $\lVert w \rVert$** subject to every point being on
the right side:

$$ \min_{w,b}\; \tfrac{1}{2}\lVert w\rVert^2 \quad\text{subject to}\quad y_i(w^\top x_i + b) \ge 1 \;\;\forall i $$

That is a convex quadratic program: one optimum, no local minima, no random initialisation.

In [ ]:
# Two well-separated blobs, and several boundaries that all classify perfectly.
X_sep, y_sep = make_blobs(n_samples=60, centers=2, cluster_std=1.05,
                          center_box=(-4, 4), random_state=6)

svm = SVC(kernel="linear", C=1000).fit(X_sep, y_sep)      # huge C ~ hard margin
w, b = svm.coef_[0], svm.intercept_[0]
margin_width = 2 / np.linalg.norm(w)

xx = np.linspace(X_sep[:, 0].min() - 1, X_sep[:, 0].max() + 1, 100)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left: three arbitrary separating lines, all "perfect" on training data.
axes[0].scatter(X_sep[:, 0], X_sep[:, 1], c=y_sep, s=30, cmap="coolwarm", edgecolor="k", lw=0.3)
for slope, intercept, style in [(-0.4, 1.2, "--"), (-1.1, 0.2, ":"), (-2.2, -1.8, "-.")]:
    axes[0].plot(xx, slope * xx + intercept, style, color="gray")
axes[0].set_title("Infinitely many lines separate these\n(three shown, all 100% correct)")

# Right: the maximum-margin boundary and its corridor.
axes[1].scatter(X_sep[:, 0], X_sep[:, 1], c=y_sep, s=30, cmap="coolwarm", edgecolor="k", lw=0.3)
boundary = -(w[0] * xx + b) / w[1]
offset = 1 / w[1]
axes[1].plot(xx, boundary, "k-", lw=2, label="decision boundary")
axes[1].plot(xx, boundary + offset, "k--", lw=1)
axes[1].plot(xx, boundary - offset, "k--", lw=1)
axes[1].fill_between(xx, boundary - abs(offset), boundary + abs(offset), alpha=0.12, color="gray")
axes[1].scatter(svm.support_vectors_[:, 0], svm.support_vectors_[:, 1], s=180,
                facecolors="none", edgecolors="lime", lw=2, label="support vectors")
axes[1].set_title(f"The maximum-margin boundary\nmargin width = {margin_width:.3f}")
axes[1].legend(fontsize=8)
for ax in axes:
    ax.set_xlim(xx.min(), xx.max()); ax.grid(alpha=0.3)
fig.tight_layout(); plt.show()

print(f"w = {np.round(w, 4)},  b = {b:.4f}")
print(f"margin width = 2 / ||w|| = {margin_width:.4f}")
print(f"support vectors: {len(svm.support_)} of {len(X_sep)} training points")
print()
print("Only the circled points touch the corridor edges. Everything else is strictly")
print("inside its own side and, as 1.2 shows, is irrelevant to where the line sits.")

## 1.2 Only the support vectors matter

This is the property the model is named after, and it is stronger than it sounds.

Solving the optimisation in its **dual** form produces one coefficient $\alpha_i \ge 0$ per
training point, and the boundary is

$$ w = \sum_i \alpha_i y_i x_i $$

The KKT conditions force $\alpha_i = 0$ for every point that is strictly on the correct side
of the margin. So **the sum only runs over the points touching or violating the margin** —
the support vectors.

The consequence: delete every non-support-vector from your training set, refit, and you get
**exactly the same model**. Not similar — the same.

In [ ]:
X_sv, y_sv = make_classification(n_samples=300, n_features=2, n_informative=2,
                                 n_redundant=0, n_clusters_per_class=1, class_sep=1.2,
                                 random_state=RANDOM_STATE)

full = SVC(kernel="linear", C=1.0, tol=1e-10).fit(X_sv, y_sv)
support_only = SVC(kernel="linear", C=1.0, tol=1e-10).fit(
    X_sv[full.support_], y_sv[full.support_])

print(f"training points : {len(X_sv)}")
print(f"support vectors : {len(full.support_)}  ({len(full.support_)/len(X_sv):.0%} of the data)")
print()
print(f"w fitted on ALL data      : {np.round(full.coef_[0], 6)}")
print(f"w fitted on SVs only      : {np.round(support_only.coef_[0], 6)}")
print(f"max |difference|          : {np.abs(full.coef_ - support_only.coef_).max():.2e}")
print(f"identical predictions on all 300 points: "
      f"{np.array_equal(full.predict(X_sv), support_only.predict(X_sv))}")
print()
print("87% of the training data was thrown away and the model did not change. That is not")
print("an approximation - the discarded points had alpha = 0, so they contributed exactly")
print("nothing to w in the first place.")
print()
print("Why this matters in practice:")
print("  - the fitted model is COMPACT: prediction cost scales with the number of support")
print("    vectors, not the training set size")
print("  - it is robust to points far from the boundary - piling on more easy examples")
print("    changes nothing (contrast with logistic regression, where every point")
print("    contributes to the likelihood)")
print("  - but it is SENSITIVE to points near the boundary, which is where mislabelled")
print("    data usually lives. Every support vector is, by definition, a hard case.")

## 1.3 Real data is not separable: the soft margin and `C`

The hard-margin problem above has **no solution** if the classes overlap even slightly — the
constraints cannot all be satisfied. Since real data essentially always overlaps, the
practical formulation introduces **slack variables** $\xi_i \ge 0$ that allow violations, and
charges for them:

$$ \min_{w,b,\xi}\; \tfrac{1}{2}\lVert w\rVert^2 + C\sum_i \xi_i \quad\text{s.t.}\quad y_i(w^\top x_i + b) \ge 1 - \xi_i $$

**`C` is the price of a violation:**

| `C` | Behaviour |
|---|---|
| **Small** | Violations are cheap → wide margin, many support vectors, more training errors tolerated. **More regularised.** |
| **Large** | Violations are expensive → narrow margin, fewer support vectors, boundary contorts to fit every point. **Less regularised.** |

Note the direction — like `LogisticRegression`'s `C` and unlike `Ridge`'s `alpha`, **smaller
`C` means stronger regularisation**. It is the same parameter in the same sense: `C`
multiplies the data-fit term, not the penalty.

In [ ]:
X_soft, y_soft = make_classification(n_samples=200, n_features=2, n_informative=2,
                                     n_redundant=0, n_clusters_per_class=1,
                                     class_sep=0.9, flip_y=0.06, random_state=3)

fig, axes = plt.subplots(1, 4, figsize=(15, 3.4))
xx_g, yy_g = np.meshgrid(np.linspace(X_soft[:, 0].min()-.5, X_soft[:, 0].max()+.5, 300),
                         np.linspace(X_soft[:, 1].min()-.5, X_soft[:, 1].max()+.5, 300))
grid = np.c_[xx_g.ravel(), yy_g.ravel()]

print(f"{'C':>8} {'support vectors':>17} {'margin width':>14} {'train accuracy':>16}")
print("-" * 60)
for ax, C in zip(axes, [0.01, 0.1, 1.0, 100.0]):
    m = SVC(kernel="linear", C=C).fit(X_soft, y_soft)
    width = 2 / np.linalg.norm(m.coef_[0])
    ax.contourf(xx_g, yy_g, m.predict(grid).reshape(xx_g.shape), alpha=0.15, levels=1)
    ax.scatter(X_soft[:, 0], X_soft[:, 1], c=y_soft, s=16, cmap="coolwarm",
               edgecolor="k", lw=0.2)
    ax.scatter(m.support_vectors_[:, 0], m.support_vectors_[:, 1], s=70,
               facecolors="none", edgecolors="lime", lw=1.2)
    ax.set_title(f"C = {C}\n{len(m.support_)} SVs, margin {width:.2f}", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
    print(f"{C:>8} {len(m.support_):>17} {width:>14.3f} {m.score(X_soft, y_soft):>16.4f}")
fig.tight_layout(); plt.show()

print()
print("Watch the two columns move in opposite directions. Small C buys a wide margin by")
print("tolerating errors, and needs many support vectors to define that wide corridor.")
print("Large C narrows the corridor until it can squeeze past the awkward points.")
print()
print("A useful diagnostic: if nearly ALL your training points are support vectors, C is")
print("probably too small (or gamma too large - see 1.6). If very few are, C may be too")
print("large and you are fitting individual points.")

## 1.4 Hinge loss, and how it differs from logistic regression

The soft-margin problem can be rewritten as an unconstrained loss plus a penalty, which makes
the comparison with NB-02 exact. With $z = y_i(w^\top x_i + b)$ (the *signed margin* — positive
when correct):

$$ \min_w \; \underbrace{\sum_i \max(0,\, 1 - z_i)}_{\text{hinge loss}} \; + \; \frac{1}{2C}\lVert w\rVert^2 $$

Compare the two losses as a function of $z$:

| | Loss | Behaviour |
|---|---|---|
| **Hinge** (SVM) | $\max(0, 1-z)$ | **Exactly zero** once $z \ge 1$. Confidently-correct points contribute *nothing*. |
| **Log-loss** (logistic) | $\log(1 + e^{-z})$ | Always positive, however correct. Every point pulls a little, forever. |

That single difference explains most of the behavioural gap between the two models:

- **Hinge loss is why support vectors exist.** A point with $z > 1$ has zero loss *and* zero
  gradient, so it cannot influence $w$ — its $\alpha_i$ is 0.
- **Log-loss is why logistic regression gives calibrated probabilities.** It is a proper
  scoring rule; hinge loss is not, which is why an SVM has no natural `predict_proba` (Part 3).
- **Hinge loss is not differentiable at $z=1$**, which is why SVMs are solved by quadratic
  programming rather than plain gradient descent.

In [ ]:
z = np.linspace(-2, 3, 400)
hinge = np.maximum(0, 1 - z)
logistic = np.log(1 + np.exp(-z)) / np.log(2)          # scaled so both pass through (0, 1)
zero_one = (z < 0).astype(float)

plt.plot(z, hinge, label="hinge (SVM)", lw=2)
plt.plot(z, logistic, label="log-loss (logistic regression)", lw=2)
plt.plot(z, zero_one, label="0/1 loss (what we actually care about)", ls=":", color="gray")
plt.axvline(1, ls="--", color="crimson", lw=1)
plt.text(1.05, 1.6, "z = 1\nhinge hits zero", fontsize=8, color="crimson")
plt.xlabel("z = y * (w.x + b)   (signed margin; positive = correct)")
plt.ylabel("loss"); plt.ylim(-0.1, 2.5)
plt.title("Both are convex surrogates for the 0/1 loss - but only one reaches zero")
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.show()

print(f"{'z':>6} {'hinge':>10} {'log-loss':>12}   interpretation")
print("-" * 62)
for zi in [-1.0, 0.0, 0.5, 1.0, 2.0, 5.0]:
    print(f"{zi:>6.1f} {max(0, 1-zi):>10.4f} {np.log(1+np.exp(-zi))/np.log(2):>12.4f}   "
          f"{'misclassified' if zi < 0 else ('inside margin' if zi < 1 else 'safely correct')}")

print()
print("At z=5 the hinge loss is exactly 0.0000 and log-loss is still 0.0097. That tiny")
print("residual is the whole difference: logistic regression keeps listening to every point,")
print("an SVM stops listening once a point is safely correct.")
print()
print("Both are CONVEX UPPER BOUNDS on the 0/1 loss we actually care about, which is not")
print("differentiable and not convex. That is the trick both models are playing.")

## 1.5 The kernel trick

This is the idea worth the whole notebook.

**The setup.** Some data is not linearly separable in its own space, but becomes separable if
you map it into a higher-dimensional one. Concentric circles are the canonical example: no
line separates them in 2-D, but add a third feature $r = x_1^2 + x_2^2$ and a *plane* separates
them trivially.

**The problem.** Useful maps blow up fast. All degree-2 terms of $d$ features gives
$\binom{d+2}{2}$ dimensions — for $d=100$ that is 5,151; for degree 3 it is 176,851. Computing
$\varphi(x)$ explicitly becomes impossible, and the RBF map is *infinite*-dimensional.

**The trick.** The dual SVM never needs $\varphi(x)$ itself — it only ever needs **inner
products** $\varphi(x)^\top \varphi(z)$. For many maps that inner product has a closed form
computable in the *original* space:

$$ K(x, z) = \varphi(x)^\top \varphi(z) $$

So you get the geometry of the high-dimensional space at the cost of the low-dimensional one.
You never build $\varphi(x)$; you never even need to know what it is.

In [ ]:
# The trick, verified. For the polynomial kernel (1 + x.z)^2 in 2-D, the explicit map is
# 6-dimensional and we can write it down - so we can check the identity directly.
def explicit_phi(X):
    """The feature map whose inner product equals (1 + x.z)^2 for 2-D inputs."""
    x1, x2 = X[:, 0], X[:, 1]
    return np.column_stack([
        np.ones(len(X)),            # 1
        np.sqrt(2) * x1,            # sqrt(2) x1
        np.sqrt(2) * x2,            # sqrt(2) x2
        x1 ** 2,                    # x1^2
        np.sqrt(2) * x1 * x2,       # sqrt(2) x1 x2
        x2 ** 2,                    # x2^2
    ])


rng = np.random.default_rng(0)
X_k = rng.normal(size=(6, 2))

K_explicit = explicit_phi(X_k) @ explicit_phi(X_k).T     # map, then dot: 6-D work
K_kernel = (1 + X_k @ X_k.T) ** 2                        # kernel: 2-D work

print("explicit phi(x).phi(z)  vs  kernel (1 + x.z)^2")
print(f"  max absolute difference: {np.abs(K_explicit - K_kernel).max():.2e}   <- identical")
print(f"  explicit map dimension for 2-D input: {explicit_phi(X_k).shape[1]}")
print()
print("Now the reason it matters - how that dimension grows for degree-2:")
print(f"{'input dims d':>14} {'phi dimension':>16}")
print("-" * 32)
for d in [2, 10, 100, 1000]:
    print(f"{d:>14} {(d + 2) * (d + 1) // 2:>16,}")
print()
print("At d=1000 the explicit map has 501,501 dimensions. The kernel computes the same")
print("inner product with a 1000-element dot product and one squaring.")
print()
print("And the RBF kernel corresponds to an INFINITE-dimensional feature map - so there the")
print("explicit route is not merely expensive, it is impossible. The kernel is 3 lines.")

In [ ]:
# Concentric circles: the textbook case where a line cannot work.
X_c, y_c = make_circles(n_samples=800, noise=0.10, factor=0.45, random_state=RANDOM_STATE)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(X_c, y_c, test_size=0.3, stratify=y_c,
                                              random_state=RANDOM_STATE)

print(f"{'model':<44} {'test accuracy':>14}")
print("-" * 60)
for label, model, tr, te in [
    ("linear kernel", SVC(kernel="linear"), Xc_tr, Xc_te),
    ("polynomial kernel (degree 3)", SVC(kernel="poly", degree=3), Xc_tr, Xc_te),
    ("RBF kernel", SVC(kernel="rbf"), Xc_tr, Xc_te),
    ("linear SVM + hand-built r = x1^2 + x2^2",
     SVC(kernel="linear"),
     np.c_[Xc_tr, (Xc_tr ** 2).sum(1)], np.c_[Xc_te, (Xc_te ** 2).sum(1)]),
]:
    print(f"{label:<44} {model.fit(tr, yc_tr).score(te, yc_te):>14.4f}")

# Draw the boundaries.
xx_c, yy_c = np.meshgrid(np.linspace(-1.6, 1.6, 300), np.linspace(-1.6, 1.6, 300))
grid_c = np.c_[xx_c.ravel(), yy_c.ravel()]
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
for ax, (label, kernel) in zip(axes, [("linear", "linear"), ("poly degree 3", "poly"),
                                      ("RBF", "rbf")]):
    m = SVC(kernel=kernel).fit(Xc_tr, yc_tr)
    ax.contourf(xx_c, yy_c, m.predict(grid_c).reshape(xx_c.shape), alpha=0.2, levels=1)
    ax.scatter(Xc_te[:, 0], Xc_te[:, 1], c=yc_te, s=8, cmap="coolwarm")
    ax.set_title(f"{label}\ntest acc {m.score(Xc_te, yc_te):.3f}", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
fig.tight_layout(); plt.show()

print()
print("The linear kernel is at chance - correctly, since no line can separate concentric")
print("circles. RBF is perfect.")
print()
print("Look at the last row of the table: hand-building the single right feature and using")
print("a LINEAR SVM does just as well. That is what the kernel is doing for you - the")
print("difference is that you had to KNOW the feature was r^2, and the kernel did not.")

### The kernels you will actually use

| Kernel | $K(x,z)$ | When |
|---|---|---|
| **Linear** | $x^\top z$ | High-dimensional data ($p$ large, especially $p > n$): text, genomics. Fast, and usually enough. |
| **RBF / Gaussian** | $\exp(-\gamma\lVert x-z\rVert^2)$ | **The default.** Handles arbitrary smooth boundaries. Infinite-dimensional feature space. |
| **Polynomial** | $(\gamma\, x^\top z + r)^d$ | When interactions of a known order matter. Numerically awkward at high degree. |
| **Sigmoid** | $\tanh(\gamma\, x^\top z + r)$ | Rarely — not always a valid kernel, and usually beaten by RBF. |

**Mercer's condition** is what makes a function a valid kernel: $K$ must be symmetric and
positive semi-definite, which guarantees *some* feature map exists whose inner product it
computes. You do not need to find that map — you only need to know it exists, because that is
what keeps the optimisation convex.

## 1.6 RBF and `gamma`: a two-dimensional grid

The RBF kernel

$$ K(x, z) = \exp\!\left(-\gamma \lVert x - z\rVert^2\right) $$

measures similarity that decays with distance. **`gamma` sets how fast:**

- **Small `gamma`** — the kernel decays slowly, so every point influences a wide region. The
  boundary is smooth, almost linear. *Underfits.*
- **Large `gamma`** — influence is tiny and local, so each training point carves out its own
  neighbourhood. The boundary becomes islands around individual points. *Overfits badly.*

`C` and `gamma` interact, so they must be tuned **together** on a 2-D grid, both on log scales.
This is the one model in the series where a grid search genuinely beats tuning one parameter at
a time.

In [ ]:
X_g, y_g = make_moons(n_samples=500, noise=0.28, random_state=RANDOM_STATE)
xx_m, yy_m = np.meshgrid(np.linspace(-2, 3, 300), np.linspace(-1.5, 2, 300))
grid_m = np.c_[xx_m.ravel(), yy_m.ravel()]

fig, axes = plt.subplots(1, 4, figsize=(15, 3.4))
for ax, gamma in zip(axes, [0.01, 0.5, 10, 300]):
    m = SVC(kernel="rbf", gamma=gamma, C=1.0).fit(X_g, y_g)
    ax.contourf(xx_m, yy_m, m.predict(grid_m).reshape(xx_m.shape), alpha=0.2, levels=1)
    ax.scatter(X_g[:, 0], X_g[:, 1], c=y_g, s=10, cmap="coolwarm", edgecolor="k", lw=0.15)
    ax.set_title(f"gamma = {gamma}\n{len(m.support_)} SVs, train acc {m.score(X_g, y_g):.3f}",
                 fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
fig.tight_layout(); plt.show()

print("gamma=0.01  : almost a straight line - the kernel barely distinguishes points")
print("gamma=0.5   : a smooth curve following the real structure")
print("gamma=10    : starting to wrap around individual points")
print("gamma=300   : islands. Training accuracy near 1.0, and it has learned nothing")
print()
print("Note the support-vector count climbing with gamma. When almost every point is a")
print("support vector, the model has memorised rather than generalised - that count is a")
print("free overfitting diagnostic you get with every SVM.")

In [ ]:
# The 2-D grid, properly cross-validated.
X_grid, y_grid = make_classification(n_samples=1500, n_features=15, n_informative=6,
                                     n_redundant=3, flip_y=0.05, random_state=RANDOM_STATE)
X_grid = StandardScaler().fit_transform(X_grid)
Xg_tr, Xg_te, yg_tr, yg_te = train_test_split(X_grid, y_grid, test_size=0.3,
                                              stratify=y_grid, random_state=RANDOM_STATE)

cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
gammas = [1e-4, 1e-3, 1e-2, 1e-1, 1.0]
Cs = [0.01, 0.1, 1, 10, 100]

table = np.zeros((len(Cs), len(gammas)))
for i, C in enumerate(Cs):
    for j, gamma in enumerate(gammas):
        table[i, j] = cross_val_score(SVC(C=C, gamma=gamma), Xg_tr, yg_tr, cv=cv).mean()

print("CV accuracy over the C x gamma grid\n")
print(f"{'C \\ gamma':>10}" + "".join(f"{g:>9.0e}" for g in gammas))
print("-" * 56)
for i, C in enumerate(Cs):
    print(f"{C:>10}" + "".join(f"{v:>9.4f}" for v in table[i]))

bi, bj = np.unravel_index(table.argmax(), table.shape)
print(f"\nbest: C={Cs[bi]}, gamma={gammas[bj]} -> {table[bi, bj]:.4f}")
print()
print("Read the shape, not just the maximum. There is a diagonal RIDGE of good values -")
print("raising C and lowering gamma trade off against each other, because both control how")
print("closely the boundary can follow individual points.")
print()
print("That ridge is exactly why you cannot tune these one at a time: the best gamma")
print("depends on which C you fixed, and vice versa. Both extremes fail - the whole")
print("C=0.01 row is at chance, and the gamma=1.0 column collapses.")

## 1.7 Scaling is not optional

Both the RBF kernel ($\lVert x - z\rVert^2$) and the linear one ($x^\top z$) are built from
**distances and dot products**. A feature measured in thousands dominates both, so a feature
in units of 0–1 contributes essentially nothing.

This is the same argument as for regularised linear models (NB-01 §1.10) and k-NN, and the
opposite of trees (NB-03 §1.6), which only compare a feature to a threshold and are exactly
scale-invariant.

**Always put a scaler in the pipeline.** This is the single most common reason an SVM
"doesn't work".

In [ ]:
X_bc, y_bc = load_breast_cancer(return_X_y=True)
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(X_bc, y_bc, test_size=0.3, stratify=y_bc,
                                              random_state=RANDOM_STATE)

feature_ranges = X_bc.max(axis=0) - X_bc.min(axis=0)
print(f"feature ranges in this dataset: smallest {feature_ranges.min():.4f}, "
      f"largest {feature_ranges.max():.1f}")
print(f"ratio: {feature_ranges.max() / feature_ranges.min():,.0f}x\n")

print(f"{'model':<40} {'test accuracy':>14}")
print("-" * 56)
for label, model in [
    ("SVC on raw features", SVC()),
    ("SVC + StandardScaler", make_pipeline(StandardScaler(), SVC())),
    ("SVC + MinMaxScaler", make_pipeline(MinMaxScaler(), SVC())),
    ("random forest on raw features (contrast)",
     RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)),
]:
    print(f"{label:<40} {model.fit(Xb_tr, yb_tr).score(Xb_te, yb_te):>14.4f}")

print()
print("Scaling is worth more here than any hyperparameter in this notebook. Meanwhile the")
print("random forest on the same raw features is untroubled - it never computes a distance.")
print()
print("The scaler must live INSIDE the pipeline so it is refitted on each CV fold")
print("(Foundations Part 4). Fitting it once on all the data before splitting is the")
print("leakage bug from Foundations 3.1.")

---
# Part 2 - Worked example: handwritten digits

1,797 images, 8×8 pixels each, ten classes. This is deliberately the kind of problem SVMs are
*still* good at: moderate sample size, relatively high dimension (64 features), dense numeric
data, and a genuinely non-linear boundary.

The workflow itself is Foundations Part 1; we spend the time on what is SVM-specific.

In [ ]:
# load_digits emits a NumPy 2.5 DeprecationWarning from inside sklearn 1.9 (it sets
# .shape on an array). Nothing to do with our code, and harmless - suppressed here so
# the notebook's output stays readable.
with warnings.catch_warnings():
    warnings.simplefilter("ignore", DeprecationWarning)
    digits = load_digits()

X_dig, y_dig = digits.data, digits.target
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(
    X_dig, y_dig, test_size=0.25, stratify=y_dig, random_state=RANDOM_STATE)

print(f"{X_dig.shape[0]} images, {X_dig.shape[1]} pixels each, "
      f"{len(np.unique(y_dig))} classes")
print(f"pixel values range {X_dig.min():.0f} to {X_dig.max():.0f}")
print(f"train {len(Xd_tr)}  test {len(Xd_te)}")

fig, axes = plt.subplots(2, 8, figsize=(11, 3))
for ax, idx in zip(axes.ravel(), range(16)):
    ax.imshow(digits.images[idx], cmap="gray_r")
    ax.set_title(str(y_dig[idx]), fontsize=9)
    ax.axis("off")
fig.suptitle("The raw data: 8x8 grids of pixel intensities", fontsize=10)
fig.tight_layout(); plt.show()

dummy = DummyClassifier(strategy="most_frequent").fit(Xd_tr, yd_tr)
print(f"\nbaseline (most frequent class): {dummy.score(Xd_te, yd_te):.4f}")

In [ ]:
cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)

candidates = {
    "SVC linear":            make_pipeline(StandardScaler(), SVC(kernel="linear")),
    "SVC poly (degree 3)":   make_pipeline(StandardScaler(), SVC(kernel="poly", degree=3)),
    "SVC RBF (defaults)":    make_pipeline(StandardScaler(), SVC()),
    "SVC RBF + PCA(40)":     make_pipeline(StandardScaler(),
                                           PCA(n_components=40, random_state=RANDOM_STATE),
                                           SVC()),
    "logistic regression":   make_pipeline(StandardScaler(),
                                           LogisticRegression(max_iter=5000)),
    "random forest":         RandomForestClassifier(n_estimators=300,
                                                    random_state=RANDOM_STATE, n_jobs=-1),
}

print(f"{'model':<26} {'CV accuracy':>16} {'fit time (s)':>14}")
print("-" * 60)
for label, model in candidates.items():
    res = cross_validate(model, Xd_tr, yd_tr, cv=cv, scoring="accuracy")
    print(f"{label:<26} {res['test_score'].mean():>8.4f} +/-{res['test_score'].std():<6.4f}"
          f" {res['fit_time'].mean():>13.3f}")

print()
print("The RBF SVM wins, and PCA to 40 components makes it slightly better AND faster -")
print("worth noting, because it is the opposite of what people expect from throwing away")
print("24 of 64 dimensions. The discarded components were mostly noise, and the kernel")
print("computes distances, which noise inflates.")
print()
print("The polynomial kernel is clearly worse here. That is typical: `poly` needs its degree")
print("and coef0 tuned to be competitive, while RBF's defaults are usually reasonable.")

In [ ]:
# Tune C and gamma together on the 2-D grid, as 1.6 argued.
grid = GridSearchCV(
    make_pipeline(StandardScaler(), SVC()),
    param_grid={
        "svc__C": [0.1, 1, 10, 100],
        "svc__gamma": ["scale", 1e-4, 1e-3, 1e-2],
    },
    cv=cv, scoring="accuracy", n_jobs=-1,
).fit(Xd_tr, yd_tr)

print("best parameters:", grid.best_params_)
print(f"best CV accuracy: {grid.best_score_:.4f}")
print(f"default CV accuracy: "
      f"{cross_val_score(make_pipeline(StandardScaler(), SVC()), Xd_tr, yd_tr, cv=cv).mean():.4f}")

results = pd.DataFrame(grid.cv_results_)
pivot = results.pivot_table(index="param_svc__C", columns="param_svc__gamma",
                            values="mean_test_score")
print("\nthe grid:")
print(pivot.round(4).to_string())
print()
print("sklearn's default gamma='scale' means 1 / (n_features * X.var()), which adapts to the")
print("data - it is why SVC's defaults are decent as long as you have scaled your features.")

In [ ]:
best_svm = grid.best_estimator_
test_pred = best_svm.predict(Xd_te)

print(f"test accuracy: {accuracy_score(yd_te, test_pred):.4f}\n")
print(classification_report(yd_te, test_pred, digits=3))

cm = confusion_matrix(yd_te, test_pred)
fig, ax = plt.subplots(figsize=(5.5, 4.6))
im = ax.imshow(cm, cmap="Blues")
ax.set_xlabel("predicted"); ax.set_ylabel("actual"); ax.set_title("Confusion matrix")
ax.set_xticks(range(10)); ax.set_yticks(range(10))
for i in range(10):
    for j in range(10):
        if cm[i, j]:
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=7,
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
fig.colorbar(im, ax=ax, fraction=0.046); fig.tight_layout(); plt.show()

errors = np.where(test_pred != yd_te)[0]
print(f"{len(errors)} misclassified out of {len(yd_te)}")
if len(errors):
    fig, axes = plt.subplots(1, min(8, len(errors)), figsize=(11, 1.8))
    for ax, idx in zip(np.atleast_1d(axes), errors[:8]):
        ax.imshow(Xd_te[idx].reshape(8, 8), cmap="gray_r")
        ax.set_title(f"{yd_te[idx]}->{test_pred[idx]}", fontsize=8)
        ax.axis("off")
    fig.suptitle("The mistakes (actual -> predicted)", fontsize=9)
    fig.tight_layout(); plt.show()
print()
print("Look at the errors as images rather than as a number. Most are genuinely ambiguous")
print("digits that a person would hesitate over - which tells you the model is near the")
print("ceiling this data supports, and that more tuning will not help.")

In [ ]:
# How compact is the fitted model?
fitted_svc = best_svm.named_steps["svc"]
n_sv = fitted_svc.support_.sum() if fitted_svc.support_.ndim == 0 else len(fitted_svc.support_)

print(f"training points          : {len(Xd_tr)}")
print(f"support vectors retained : {n_sv}  ({n_sv / len(Xd_tr):.0%})")
print(f"support vectors per class: {fitted_svc.n_support_}")
print()
print("Prediction cost is proportional to the number of support vectors, not to the training")
print("set size - the model 'compresses' the data down to the hard cases near the boundaries.")
print()
print("On this problem it keeps a large fraction, which is a sign the classes genuinely")
print("touch in 64-dimensional pixel space. On easier problems it can be a few percent")
print("(1.2 showed 13%).")

---
# Part 3 - The practical limits

SVMs lost their dominance for concrete, measurable reasons. Knowing them is what lets you
choose the model deliberately.

## 3.1 They do not scale

The dual problem involves the **kernel matrix** $K$, which is $n \times n$. That is
$O(n^2)$ memory and roughly $O(n^2)$ to $O(n^3)$ time. At 100,000 rows the kernel matrix alone
is 80 GB.

Practical implementations (libsvm, which sklearn's `SVC` wraps) use decomposition methods that
avoid materialising all of $K$, but the fundamental scaling remains.

In [ ]:
sizes = [1000, 2000, 4000, 8000, 16000]
svc_times, linsvc_times, sgd_times = [], [], []

for n in sizes:
    Xn, yn = make_classification(n_samples=n, n_features=20, n_informative=10,
                                 random_state=RANDOM_STATE)
    Xn = StandardScaler().fit_transform(Xn)
    for store, model in [(svc_times, SVC()),
                         (linsvc_times, LinearSVC(max_iter=5000)),
                         (sgd_times, SGDClassifier(max_iter=1000, random_state=RANDOM_STATE))]:
        t0 = time.time()
        model.fit(Xn, yn)
        store.append(time.time() - t0)

def scaling_exponent(times):
    """Fit time ~ n^p; return p."""
    return np.polyfit(np.log(sizes), np.log(times), 1)[0]


svc_exp = scaling_exponent(svc_times)
lin_exp = scaling_exponent(linsvc_times)

print(f"{'n':>8} {'SVC (RBF)':>12} {'LinearSVC':>12} {'SGDClassifier':>15}")
print("-" * 52)
for n, a, b, c in zip(sizes, svc_times, linsvc_times, sgd_times):
    print(f"{n:>8} {a:>12.3f} {b:>12.4f} {c:>15.4f}")

print(f"\nfitted scaling:  SVC ~ n^{svc_exp:.2f}     LinearSVC ~ n^{lin_exp:.2f}")
print(f"16x more data cost SVC {svc_times[-1]/svc_times[0]:.0f}x more time "
      f"({svc_times[-1]:.2f}s at n={sizes[-1]:,})")

plt.loglog(sizes, svc_times, marker="o", label="SVC (kernel, RBF)")
plt.loglog(sizes, linsvc_times, marker="s", label="LinearSVC")
plt.loglog(sizes, sgd_times, marker="^", label="SGDClassifier (hinge)")
plt.xlabel("training rows"); plt.ylabel("fit time (s)")
plt.title("Kernel SVMs scale super-linearly - and the gap keeps widening")
plt.legend(fontsize=8); plt.grid(alpha=0.3, which="both"); plt.show()

target = 500_000
projected = svc_times[-1] * (target / sizes[-1]) ** svc_exp
print()
print(f"Extrapolating that exponent: the {svc_times[-1]:.1f}s fit at {sizes[-1]:,} rows becomes")
print(f"roughly {projected/60:.0f} minutes at {target:,} rows - for ONE fit. Multiply by a")
print("5-fold CV over a C x gamma grid and it is days.")
print()
print("A caveat on reading the linear models' column honestly: their times are in")
print("milliseconds, so fixed overheads (validation, memory allocation) dominate and their")
print("measured exponents are inflated. The signal to trust is the SVC column and the")
print("widening GAP, not the precise exponent of a 3-millisecond fit.")
print()
print("The practical rule: above roughly 10,000-50,000 rows, stop using a kernel SVM.")

### What to use instead, above ~50k rows

| Option | Idea |
|---|---|
| `LinearSVC` | Solves the *primal* directly. Linear only, but scales to millions of rows. |
| `SGDClassifier(loss="hinge")` | Stochastic gradient on the same hinge loss. Streams, supports `partial_fit`. |
| **`Nystroem` / `RBFSampler` + linear model** | **Approximate the kernel explicitly.** Builds a finite feature map whose dot product approximates the RBF kernel, then fits a fast linear model on it. Kernel-ish accuracy at linear cost. |
| Gradient boosting (NB-05) | Usually the better answer on large tabular data anyway. |

The third one is worth knowing about — it is the kernel trick run in reverse, and it turns a
quadratic problem back into a linear one.

In [ ]:
X_big, y_big = make_classification(n_samples=12000, n_features=20, n_informative=10,
                                   n_redundant=4, random_state=RANDOM_STATE)
X_big = StandardScaler().fit_transform(X_big)
Xb_tr2, Xb_te2, yb_tr2, yb_te2 = train_test_split(X_big, y_big, test_size=0.25,
                                                  stratify=y_big, random_state=RANDOM_STATE)

# IMPORTANT: the kernel approximators do NOT inherit SVC's gamma="scale" default - they
# default to gamma=1.0. On scaled data with p features, "scale" means 1/(p * var), so we
# compute it and pass it explicitly. Getting this wrong is the difference between the last
# two rows of the table below.
gamma_scale = 1.0 / (X_big.shape[1] * X_big.var())
print(f"{len(Xb_tr2):,} training rows;  gamma='scale' equivalent = {gamma_scale:.4f} "
      f"(the approximators default to 1.0)\n")

print(f"{'model':<48} {'seconds':>9} {'test accuracy':>15}")
print("-" * 76)
for label, model in [
    ("SVC (exact RBF kernel)", SVC()),
    ("LinearSVC", LinearSVC(max_iter=5000)),
    ("Nystroem(300, gamma=scale) + LinearSVC", make_pipeline(
        Nystroem(gamma=gamma_scale, n_components=300, random_state=RANDOM_STATE),
        LinearSVC(max_iter=5000))),
    ("RBFSampler(1000, gamma=scale) + SGD", make_pipeline(
        RBFSampler(gamma=gamma_scale, n_components=1000, random_state=RANDOM_STATE),
        SGDClassifier(max_iter=2000, random_state=RANDOM_STATE))),
    ("RBFSampler(300, DEFAULT gamma=1.0) + SGD", make_pipeline(
        RBFSampler(n_components=300, random_state=RANDOM_STATE),
        SGDClassifier(max_iter=2000, random_state=RANDOM_STATE))),
]:
    t0 = time.time()
    model.fit(Xb_tr2, yb_tr2)
    elapsed = time.time() - t0
    print(f"{label:<48} {elapsed:>9.2f} {model.score(Xb_te2, yb_te2):>15.4f}")

print()
print("Read the table from the top:")
print()
print("  SVC exact          - the target we are trying to approximate cheaply.")
print("  LinearSVC alone    - clearly worse. This data is NOT linearly separable, which is")
print("                       exactly why the kernel was worth having.")
print("  Nystroem + linear  - recovers nearly all of the kernel's advantage, faster.")
print("  RBFSampler + SGD   - also works, and is the cheapest of the lot.")
print()
print("Now the LAST row, which is the one to remember. It is the same method as the row")
print("above it, with gamma left at the library default of 1.0 instead of matching SVC's")
print("'scale'. It collapses to near chance.")
print()
print("Nystroem and RBFSampler do NOT inherit SVC's gamma='scale' default. If you swap a")
print("kernel SVM for an approximation and the accuracy falls off a cliff, this is almost")
print("always why - and nothing warns you, because a badly-chosen gamma is not an error.")

## 3.2 An SVM has no probabilities — and the API that pretended otherwise is being removed

An SVM's `decision_function` returns a **signed distance from the boundary**: unbounded, in
arbitrary units. There is no probability hiding in it to extract, and that is not an
implementation gap — hinge loss is **not a proper scoring rule** (§1.4), so nothing in the
objective ever asked for a calibrated number.

For years the workaround was `SVC(probability=True)`, which quietly ran 5-fold internal
cross-validation and fitted **Platt scaling** (a 1-D logistic regression) on the decision
values. It had a genuine wart: `predict()` used the sign of the decision function while
`predict_proba()` used the separately-fitted Platt model, so **the two could disagree**.

> **As of scikit-learn 1.9, `probability=True` is deprecated** and will be removed in 1.11.
> The replacement is explicit: `CalibratedClassifierCV(SVC(), ensemble=False)`.

That is a good change — the calibration was always a separate model bolted on afterwards, and
now the API says so.

In [ ]:
X_p, y_p = make_classification(n_samples=3000, n_features=20, n_informative=8,
                               random_state=RANDOM_STATE)
X_p = StandardScaler().fit_transform(X_p)
Xp_tr, Xp_te, yp_tr, yp_te = train_test_split(X_p, y_p, test_size=0.3, stratify=y_p,
                                              random_state=RANDOM_STATE)

t0 = time.time(); plain = SVC().fit(Xp_tr, yp_tr); t_plain = time.time() - t0
t0 = time.time()
calibrated_svc = CalibratedClassifierCV(SVC(), ensemble=False, cv=5).fit(Xp_tr, yp_tr)
t_calib = time.time() - t0

print(f"SVC()                                          {t_plain:.3f}s")
print(f"CalibratedClassifierCV(SVC(), ensemble=False)  {t_calib:.3f}s   "
      f"({t_calib/t_plain:.1f}x slower)")
print()
print("The cost is unchanged by the API move: you are still fitting the SVM once per CV")
print("fold plus a final refit. Probabilities from an SVM are never free.")

decision = plain.decision_function(Xp_te)                  # the raw, uncalibrated score
proba = calibrated_svc.predict_proba(Xp_te)[:, 1]          # the calibrated probability

shifted = (proba >= 0.5).astype(int) != (decision > 0).astype(int)
internal = (calibrated_svc.predict(Xp_te) != (proba >= 0.5).astype(int)).sum()

print(f"\nrows where 'calibrated p >= 0.5' differs from 'decision_function > 0': "
      f"{shifted.sum()} of {len(yp_te)}  ({shifted.mean():.2%})")
print(f"rows where the calibrated model's own predict() and predict_proba() disagree: "
      f"{internal}")
print(f"\nAUC from raw decision_function : {roc_auc_score(yp_te, decision):.4f}")
print(f"AUC from calibrated probability: {roc_auc_score(yp_te, proba):.4f}")
print()
print("Three things in those numbers:")
print()
print("1. The AUCs are essentially identical, because Platt scaling is MONOTONIC - it")
print("   rescales the decision values without reordering them. Calibration cannot improve")
print("   ranking, and ranking metrics cannot detect that it happened (NB-04 Q11).")
print()
print("2. The 0.5 CUT still lands somewhere different from the decision function's sign, on")
print("   a couple of percent of rows. That is the calibration doing its job - the raw")
print("   boundary is not where a 50% probability actually sits.")
print()
print("3. The new API is INTERNALLY consistent: its own predict() and predict_proba() agree")
print("   exactly. That inconsistency was one of the reasons the old parameter was retired.")
print()
print("What to do:")
print("  - need a RANKING or a threshold? Use decision_function and choose the threshold")
print("    yourself (NB-02 Part 3). No calibration, no 6x cost.")
print("  - need real probabilities? CalibratedClassifierCV, deliberately, and pick")
print("    'sigmoid' (Platt, robust on small data) or 'isotonic' (flexible, needs more).")
print("  - are probabilities central to the problem? Logistic regression produces them by")
print("    construction, at a fraction of the cost (NB-02).")

In [ ]:
# Sigmoid vs isotonic, against a model that is calibrated by construction.
isotonic = CalibratedClassifierCV(SVC(), method="isotonic", cv=5).fit(Xp_tr, yp_tr)
logistic = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(Xp_tr, yp_tr)

fig, ax = plt.subplots(figsize=(5.4, 4.2))
ax.plot([0, 1], [0, 1], "--", color="gray", label="perfectly calibrated")
print(f"{'model':<40} {'Brier':>8} {'log-loss':>10} {'AUC':>8}")
print("-" * 68)
for label, p in [("SVC + sigmoid (Platt)", proba),
                 ("SVC + isotonic", isotonic.predict_proba(Xp_te)[:, 1]),
                 ("logistic regression (calibrated by design)",
                  logistic.predict_proba(Xp_te)[:, 1])]:
    frac_pos, mean_pred = calibration_curve(yp_te, p, n_bins=8, strategy="quantile")
    ax.plot(mean_pred, frac_pos, marker="o", ms=4, label=label[:28])
    print(f"{label:<40} {brier_score_loss(yp_te, p):>8.4f} "
          f"{log_loss(yp_te, p):>10.4f} {roc_auc_score(yp_te, p):>8.4f}")
ax.set(xlabel="predicted probability", ylabel="observed frequency",
       title="An SVM's probabilities are fitted on top, not produced by it")
ax.legend(fontsize=7); ax.grid(alpha=0.3); fig.tight_layout(); plt.show()

print()
print("All three land close together, which is the honest result - Platt scaling works, and")
print("on 2,100 training rows isotonic has enough data not to overfit.")
print()
print("The point is not that one wins. It is that for the SVM the probability is a SEPARATE")
print("model fitted on top of the decision values, while for logistic regression it is what")
print("the optimisation was maximising in the first place. When someone asks where the")
print("number came from, those are different answers.")

## 3.3 Multiclass, and the other SVM variants

**Multiclass.** The SVM formulation is inherently binary. sklearn's `SVC` uses
**one-vs-one**: it fits $\binom{k}{2}$ binary classifiers and votes. For 10 digit classes that
is 45 models — though each is trained on only the two relevant classes, so it is often faster
than one-vs-rest on large $k$. `LinearSVC` uses one-vs-rest instead.

**The family:**

| Class | Use |
|---|---|
| `SVC` | Kernel classification. The default subject of this notebook. |
| `LinearSVC` | Linear only, primal solver, scales far better (§3.1) |
| `SVR` / `LinearSVR` | **Regression.** Same idea inverted: fit a tube of width $\varepsilon$ and penalise points *outside* it |
| `OneClassSVM` | **Novelty detection** — learn a boundary around "normal" with no labels at all (revisited in NB-15) |
| `NuSVC` / `NuSVR` | Reparameterised: `nu` directly bounds the fraction of support vectors |

In [ ]:
print(f"SVC multiclass strategy: {SVC().fit(Xd_tr[:200], yd_tr[:200]).decision_function_shape}")
print(f"binary classifiers fitted for {len(np.unique(y_dig))} classes (one-vs-one): "
      f"{len(np.unique(y_dig)) * (len(np.unique(y_dig)) - 1) // 2}")

# SVR: the epsilon-tube idea, visualised.
rng = np.random.default_rng(1)
x_r = np.sort(rng.uniform(0, 6, 120)).reshape(-1, 1)
y_r = np.sin(x_r.ravel()) * 3 + rng.normal(0, 0.4, 120)

svr = SVR(kernel="rbf", C=10, epsilon=0.5, gamma=0.5).fit(x_r, y_r)
smooth = np.linspace(0, 6, 300).reshape(-1, 1)
pred = svr.predict(smooth)

plt.scatter(x_r, y_r, s=14, alpha=0.5, label="data")
plt.plot(smooth, pred, color="crimson", lw=2, label="SVR prediction")
plt.fill_between(smooth.ravel(), pred - 0.5, pred + 0.5, alpha=0.15, color="crimson",
                 label="epsilon tube (+/- 0.5)")
plt.scatter(x_r[svr.support_], y_r[svr.support_], s=70, facecolors="none",
            edgecolors="lime", lw=1.2, label="support vectors")
plt.xlabel("x"); plt.ylabel("y"); plt.title("SVR: fit a tube, penalise only what falls outside")
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.show()

print(f"\nSVR kept {len(svr.support_)} of {len(x_r)} points as support vectors")
print()
print("The logic inverts neatly. Classification: points OUTSIDE the margin are free, points")
print("inside or misclassified cost you. Regression: points INSIDE the epsilon tube are")
print("free, points outside cost you. In both cases the free points have alpha = 0 and")
print("drop out of the model entirely.")

---
# Part 4 - Tough questions

---

### Q1. Logistic regression and a linear SVM both draw a straight boundary. What makes them choose *different* straight lines?

<details><summary>Answer</summary>

**Different loss functions, and therefore different notions of "best".**

- **Logistic regression** maximises likelihood under log-loss. Every point contributes to the
  gradient forever — a point far on the correct side still has a small non-zero loss
  ($\log(1+e^{-z}) > 0$ for all finite $z$), so it still pulls on $w$.
- **An SVM** minimises hinge loss, which is **exactly zero** once $z \ge 1$. Confidently
  correct points contribute nothing at all, so the boundary is determined only by the points
  near it — the support vectors.

Practical consequences:

- **Adding easy examples changes logistic regression and does not change an SVM.** §1.2 shows
  87% of the data being deleted with no effect on $w$.
- **Outliers far on the correct side** barely move an SVM; they do move logistic regression.
- **Points near the boundary** dominate the SVM completely — which is where mislabelled data
  usually lives, so SVMs are sensitive to label noise near the margin.
- **Logistic regression gives calibrated probabilities**; the SVM does not, because hinge loss
  is not a proper scoring rule (Q9).

Empirically they often land in similar places on clean, separable-ish data. They diverge when
the classes overlap heavily or when there are outliers.

</details>

---

### Q2. What exactly is a support vector, and why can you delete every other training point?

<details><summary>Answer</summary>

Solving the SVM in its **dual** form gives one coefficient $\alpha_i \ge 0$ per training
point, and the boundary is $w = \sum_i \alpha_i y_i x_i$. The KKT complementary-slackness
conditions force

$$ \alpha_i = 0 \quad\text{for every point strictly outside the margin on the correct side} $$

A **support vector** is a point with $\alpha_i > 0$ — one lying *on* the margin boundary,
inside the margin, or misclassified.

So the sum defining $w$ only runs over the support vectors. Every other point multiplies by
zero. Deleting them cannot change $w$, and §1.2 verifies it: refitting on only the 13% that
were support vectors gives $w$ identical to $2.9\times10^{-11}$ and **identical predictions**.

**Why it matters:**

- **Compact models.** Prediction cost scales with the number of support vectors, not the
  training set size. The SVM effectively compresses the data to its hard cases.
- **A free overfitting diagnostic.** If nearly every point is a support vector, `C` is too
  small or `gamma` too large — the model is memorising (§1.6).
- **The flip side:** every support vector is by definition a *hard case*, and hard cases are
  where mislabelled data lives. An SVM listens hardest to exactly the rows most likely to be
  wrong.

</details>

---

### Q3. Explain the kernel trick to someone who knows linear algebra but not ML. Why is it a "trick" rather than just a feature transformation?

<details><summary>Answer</summary>

**The setup.** Some data is not linearly separable in its own space but becomes separable in a
higher-dimensional one. So map each point through some $\varphi$ and fit a linear model there.

**The obstacle.** Useful maps explode. All degree-2 terms of $d$ features needs
$\binom{d+2}{2}$ dimensions — 5,151 for $d=100$, 501,501 for $d=1000$ (§1.5 prints this). The
RBF kernel's map is **infinite**-dimensional. Computing $\varphi(x)$ is impractical or
impossible.

**The trick.** The dual SVM never uses $\varphi(x)$ on its own — it only ever needs **inner
products** $\varphi(x)^\top\varphi(z)$. For well-chosen maps that inner product has a closed
form evaluable in the *original* space:

$$ K(x,z) = \varphi(x)^\top \varphi(z) $$

For example $(1 + x^\top z)^2$ equals the inner product of a specific 6-dimensional map when
$d=2$ — §1.5 verifies this to $4\times10^{-16}$.

**Why "trick" and not just "a transformation":** you never perform the transformation. You get
the *geometry* of a 501,501-dimensional space while doing arithmetic in 1,000 dimensions. With
an RBF kernel you operate in an infinite-dimensional space using a three-line formula.

**The catch:** it only works for algorithms expressible purely in terms of inner products
between data points. That is why "kernelised" versions exist for SVMs, PCA, ridge regression
and Gaussian processes, but not for decision trees.

**Mercer's condition** is the requirement: $K$ symmetric and positive semi-definite guarantees
*some* $\varphi$ exists. You never need to find it — only to know it does, because that keeps
the problem convex.

</details>

---

### Q4. `C` and `gamma` both control overfitting. Why must they be tuned together rather than one at a time?

<details><summary>Answer</summary>

They control **different mechanisms** that produce a similar effect, so they trade off against
each other and the best value of one depends on the other.

- **`C`** is the price of violating the margin. Large `C` → the boundary contorts to classify
  every training point.
- **`gamma`** is how fast RBF similarity decays with distance. Large `gamma` → each point
  influences only its immediate neighbourhood, so the boundary can form islands.

Either alone can produce a wiggly overfitted boundary, and they can partly **cancel**: a large
`gamma` (locally flexible) with a small `C` (violations cheap) may generalise fine, while
either extreme with the other held at a bad value fails.

§1.6's grid shows this directly — a diagonal **ridge** of good values rather than a single
peak, with the whole `C=0.01` row at chance and the `gamma=1.0` column collapsing.

**Practical procedure:**

1. **Scale your features first** (§1.7) — `gamma`'s meaning depends entirely on the scale of
   $\lVert x-z\rVert^2$.
2. Grid over **both**, on **log scales**: `C` in `np.logspace(-2, 3, 6)`, `gamma` in
   `np.logspace(-4, 1, 6)` or `"scale"`.
3. This is the one model in the series where an exhaustive 2-D grid genuinely beats random
   search, because there are only two parameters and the interaction is strong.

</details>

---

### Q5. You fit an SVM and it performs terribly. What is the first thing you check?

<details><summary>Answer</summary>

**Whether you scaled the features.** It is by a wide margin the most common cause, and §1.7
measures it: 0.906 → 0.977 on breast cancer from adding a `StandardScaler`.

The reason is structural, not incidental: both the linear kernel ($x^\top z$) and the RBF
kernel ($\exp(-\gamma\lVert x-z\rVert^2)$) are built from **dot products and distances**. A
feature ranging over thousands dominates both sums; a feature ranging 0–1 contributes almost
nothing. The model is effectively fitted on your largest-magnitude column.

It also breaks `gamma`: the default `"scale"` adapts to overall variance, but per-feature
scale differences remain, and any `gamma` you tune is only meaningful for the scaling you
tuned it on.

**The rest of the checklist, in order:**

2. Is the scaler **inside a Pipeline**, so it refits per CV fold? (Foundations §3.1)
3. Are `C` and `gamma` tuned **together** on log scales (Q4)? The defaults are only sometimes
   right.
4. Is `n` too large — is it slow rather than bad (§3.1)?
5. Is the problem heavily imbalanced? Consider `class_weight="balanced"`.
6. Are you reading `predict_proba` from `probability=True` and confused by a disagreement with
   `predict` (§3.2)?

Contrast with a tree ensemble, where step 1 does not exist at all — NB-03 §1.6 shows trees are
*exactly* invariant to monotone rescaling.

</details>

---

### Q6. Why do SVMs stop being practical above roughly 50,000 rows, and what do you use instead?

<details><summary>Answer</summary>

The dual formulation is built on the **kernel matrix** $K$, which is $n \times n$: $O(n^2)$
memory and roughly $O(n^2)$–$O(n^3)$ time. At $n=100{,}000$, storing $K$ in float64 is 80 GB.
Real solvers (libsvm) decompose the problem to avoid materialising all of $K$, but the scaling
stands — §3.1 measures an empirical exponent near $n^{1.7}$.

Compounding it: you need to **cross-validate over a 2-D `C`×`gamma` grid** (Q4), so the fit
cost is multiplied by tens.

**The alternatives, in rough order:**

1. **`LinearSVC`** — solves the *primal*, so no kernel matrix. Linear boundaries only, but
   scales to millions of rows. Excellent on high-dimensional sparse data like text.
2. **`SGDClassifier(loss="hinge")`** — the same hinge loss by stochastic gradient descent.
   Streams, supports `partial_fit`.
3. **Kernel approximation** — `Nystroem` or `RBFSampler` build an explicit finite feature map
   whose dot product *approximates* the RBF kernel, then you fit a linear model on it. §3.1
   shows this recovering most of the kernel's advantage at a fraction of the cost. This is the
   kernel trick run in reverse.
4. **Gradient boosting (NB-05)** — on large tabular data it is usually both faster and more
   accurate, which is the honest reason SVMs faded.

**Where kernel SVMs remain the right choice:** small-to-medium $n$ with high $p$, especially
$p > n$ — text with rich features, genomics, spectroscopy. There the kernel matrix is small and
tree ensembles struggle.

</details>

---

### Q7. Hard-margin SVMs have no solution on real data. What breaks, and what fixes it?

<details><summary>Answer</summary>

The hard-margin problem requires $y_i(w^\top x_i + b) \ge 1$ for **every** point. If the
classes overlap even by one row — one mislabelled example, one genuine ambiguity — the
constraint set is **empty** and the optimisation is infeasible. There is no "best effort"; it
simply has no solution.

**The fix: slack variables.** Introduce $\xi_i \ge 0$ per point, relax the constraint to
$y_i(w^\top x_i + b) \ge 1 - \xi_i$, and add a penalty:

$$ \min \; \tfrac12\lVert w\rVert^2 + C\sum_i \xi_i $$

`C` is the price of each unit of violation, so it interpolates between "maximise margin,
tolerate errors" (small `C`) and "classify everything, margin be damned" (large `C`).

**Two things worth knowing:**

- **This is regularisation in disguise.** Rewriting as hinge loss plus $\frac{1}{2C}\lVert
  w\rVert^2$ (§1.4) makes it an ordinary regularised loss minimisation, with `1/C` playing the
  role of Ridge's `alpha`.
- **There is a parallel with NB-02 Q7.** Logistic regression's failure on *perfectly separable*
  data (weights → ∞) and the SVM's failure on *non-separable* data are both cases where the
  unconstrained objective has no finite optimum, and both are fixed by adding a penalty term.

</details>

---

### Q8. Someone proposes a custom similarity function as a kernel. What must you check?

<details><summary>Answer</summary>

**Mercer's condition:** $K$ must be **symmetric** ($K(x,z) = K(z,x)$) and **positive
semi-definite** — for any finite set of points, the Gram matrix $K_{ij} = K(x_i, x_j)$ must
have no negative eigenvalues.

**Why it matters:** PSD is precisely the condition guaranteeing that some feature map
$\varphi$ exists with $K(x,z) = \varphi(x)^\top\varphi(z)$. That is what makes the dual problem
a **convex** quadratic program with a unique solution. Feed in a non-PSD "kernel" and you have
a non-convex problem: the solver may not converge, may return something meaningless, and the
"margin" interpretation evaporates.

**How to check in practice:** build the Gram matrix on a sample and inspect its eigenvalues —
`np.linalg.eigvalsh(K).min()` should be ≥ 0 up to numerical tolerance.

**Useful closure properties** (so you can build valid kernels from valid ones): if $K_1$ and
$K_2$ are kernels, so are $K_1 + K_2$, $cK_1$ for $c>0$, $K_1 K_2$, and $f(x)K_1(x,z)f(z)$.
These let you compose, for example, a text kernel and a numeric kernel.

**Note the famous exception:** the `sigmoid` kernel $\tanh(\gamma x^\top z + r)$ is **not** PSD
for all parameter values, yet sklearn offers it and it sometimes works. That is empirical luck,
not theory — and a reason to prefer RBF.

</details>

---

### Q9. Why does an SVM not naturally produce probabilities, and what is the right way to get them?

<details><summary>Answer</summary>

**Why not:** `decision_function` returns a signed *distance* from the boundary — unbounded, in
arbitrary units. It ranks well but is not on a probability scale. More fundamentally, hinge
loss is **not a proper scoring rule**: it is minimised by getting the *sign and margin* right,
never by getting a *probability* right. Log-loss **is** a proper scoring rule, which is exactly
why logistic regression's outputs are calibrated by construction (NB-02).

**The historical answer was `SVC(probability=True)`**, which ran 5-fold internal
cross-validation and fitted **Platt scaling** — a 1-D logistic regression mapping decision
values to probabilities. It had a real wart: `predict()` used the sign of the decision
function while `predict_proba()` used the Platt model, so the two could disagree on a couple of
percent of rows.

**As of scikit-learn 1.9 that parameter is deprecated**, removal scheduled for 1.11. The
replacement is `CalibratedClassifierCV(SVC(), ensemble=False)` — the same computation, made
explicit, and internally consistent.

**What §3.2 measures:**

1. **The cost is unchanged: ~6× a plain fit.** You are still fitting the SVM once per fold plus
   a refit. Probabilities from an SVM are never free.
2. **AUC is unchanged**, because Platt scaling is monotonic — it rescales without reordering.
   Calibration cannot improve ranking, and ranking metrics cannot detect it happened.
3. **The 0.5 cut still moves** relative to the raw decision-function sign, on ~2.4% of rows.
   That is calibration working: the geometric boundary is not where a 50% probability sits.

**What to do:**

- Only need ranking or a threshold? Use `decision_function`, set the threshold from costs
  (NB-02 Part 3), and skip calibration entirely.
- Genuinely need probabilities? `CalibratedClassifierCV` deliberately — `sigmoid` for small
  data, `isotonic` when you have a few thousand rows.
- Probabilities central to the problem? Use logistic regression and stop paying the 6×.

</details>

---

### Q10. Give three problem types where a kernel SVM is still the right choice in 2020s practice.

<details><summary>Answer</summary>

1. **$p > n$ problems** — more features than samples. Genomics (20,000 genes, 200 patients),
   spectroscopy, some medical imaging. The kernel matrix is $n\times n$ so it stays *small*
   exactly when $n$ is small, and the margin-maximisation principle is a genuinely good
   inductive bias when you cannot afford to estimate many parameters. Tree ensembles struggle
   here because random feature subsets rarely contain the informative ones.

2. **High-dimensional sparse text with a linear kernel** — `LinearSVC` on TF-IDF remains a
   very strong, very fast baseline for document classification, and often beats gradient
   boosting, which handles wide sparse data poorly (NB-04 Q9).

3. **Small, clean, dense numeric datasets with a genuinely curved boundary** — a few hundred to
   a few thousand rows. Boosting has too many hyperparameters to tune reliably on that little
   data; an SVM has two, and RBF handles smooth non-linearity gracefully. Part 2's digits
   example is exactly this shape.

**A fourth, different in kind:** `OneClassSVM` for **novelty detection** when you have only
"normal" examples and no labelled anomalies — revisited in NB-15.

**And the honest limit:** for medium-to-large tabular data with mixed types, missing values and
categoricals, gradient boosting wins on accuracy, speed and convenience. That is why SVMs
faded, and pretending otherwise helps nobody.

</details>

---

### Q11. How do SVMs handle multiclass, and what is the cost?

<details><summary>Answer</summary>

The formulation is **inherently binary** — margin between *two* classes. Multiclass is bolted
on by decomposition.

**One-vs-one (sklearn's `SVC`):** fit $\binom{k}{2}$ classifiers, one per class pair, and vote.
For 10 digit classes that is **45 models**.

**One-vs-rest (`LinearSVC`):** fit $k$ classifiers, each "class $i$ vs everything else".

**The counter-intuitive part:** one-vs-one is often *faster* despite fitting far more models,
because each is trained on only the rows from its two classes. With SVM's super-linear scaling
in $n$ (Q6), many small problems beat a few large ones — $\binom{k}{2}$ fits on $2n/k$ rows
each is cheaper than $k$ fits on $n$ rows when the cost grows like $n^2$.

**Costs:**

- **Interpretation gets murky.** `decision_function` returns pairwise vote aggregations, not
  clean per-class scores.
- **Ties are possible** in voting and are broken somewhat arbitrarily.
- **`probability=True` gets even more expensive** — Platt scaling on top of 45 models.

**Contrast:** logistic regression extends *natively* via softmax with a single joint
optimisation (NB-02 §1.7), and trees handle multiclass natively at the leaves. The
decomposition is an SVM-specific workaround.

</details>

---

### Q12. Your SVM gets 0.97 and your gradient boosting gets 0.97 on the same data. Which do you ship?

<details><summary>Answer</summary>

**First establish the tie is real** — same CV splits, compared fold by fold, difference against
the fold-to-fold standard deviation (NB-05 Q12). Assume it survives.

**Then it is an engineering decision, and the answer usually depends on the data shape.**

**Reasons to ship the SVM:**
- **Very few hyperparameters** — two, on a well-understood grid. Boosting has six that interact.
- **Compact model** — support vectors only, which on some problems is a small fraction of the
  data (§1.2 showed 13%).
- **Deterministic and convex** — one optimum, no seed sensitivity, identical model every retrain.
- Data is $p > n$, or high-dimensional and dense — the regime where SVMs are genuinely strong.

**Reasons to ship the boosting model:**
- **It will scale** as the dataset grows. The SVM will not (Q6) — a tie today at 10k rows
  becomes an infeasible fit at 500k.
- **Native handling of categoricals and missing values.** The SVM needs both engineered away.
- **No scaling dependency**, so the production pipeline is simpler and one less thing to get
  wrong.
- **Better probability estimates** without an extra calibration stage (Q9).
- **Feature importance** comes for free (with NB-04 Part 3's caveats).

**My default answer: the boosting model**, mainly for the scaling and preprocessing arguments —
you are choosing a system that will still work when the data doubles, not just the model that
tied today.

**Unless** you are in $p>n$, where the SVM is the more principled choice and the tie will
probably become a win as you add features.

</details>

---

## Coding challenges

In [ ]:
# ---------------------------------------------------------------------------
# CHALLENGE 1
# Verify the kernel trick for the RBF kernel, which has an INFINITE-dimensional
# feature map - so you cannot write phi(x) down.
#
# Instead: approximate it. Use Nystroem to build a finite feature map, and show
# that dot products in that space converge to the RBF kernel values as you add
# components.
# ---------------------------------------------------------------------------

# YOUR CODE HERE

In [ ]:
# --- Challenge 1: one solution ------------------------------------------
rng = np.random.default_rng(0)
X_ch1 = rng.normal(size=(200, 5))
GAMMA = 0.3

# The exact RBF kernel matrix.
sq_dists = ((X_ch1[:, None, :] - X_ch1[None, :, :]) ** 2).sum(axis=2)
K_exact = np.exp(-GAMMA * sq_dists)

print("approximating an INFINITE-dimensional feature map with a finite one\n")
print(f"{'n_components':>14} {'max |K_approx - K_exact|':>26} {'mean abs error':>17}")
print("-" * 60)
for n_comp in [5, 20, 50, 100, 199]:
    feature_map = Nystroem(gamma=GAMMA, n_components=n_comp,
                           random_state=RANDOM_STATE).fit(X_ch1)
    Z = feature_map.transform(X_ch1)          # our explicit, finite phi(x)
    K_approx = Z @ Z.T                        # dot products in that space
    print(f"{n_comp:>14} {np.abs(K_approx - K_exact).max():>26.6f} "
          f"{np.abs(K_approx - K_exact).mean():>17.6f}")

print()
print("The approximation converges as components are added - at 199 components on 200")
print("points it is essentially exact, because Nystroem is doing a low-rank")
print("reconstruction of the kernel matrix from a subset of the rows.")
print()
print("The point: the RBF kernel's true feature map is infinite-dimensional and can never")
print("be written down, yet a few hundred dimensions reproduce its inner products to several")
print("decimal places. That is why Nystroem + a linear model is a practical substitute for")
print("a kernel SVM on large data (Part 3.1).")

In [ ]:
# ---------------------------------------------------------------------------
# CHALLENGE 2
# Demonstrate that support vectors are the ONLY thing that matters - and that
# this makes SVMs vulnerable in a specific way.
#
# Take a clean dataset. Flip the label of ONE point far from the boundary, and
# ONE point near it. Measure how much the boundary moves in each case, and
# compare against logistic regression.
# ---------------------------------------------------------------------------

# YOUR CODE HERE

In [ ]:
# --- Challenge 2: one solution ------------------------------------------
X_ch2, y_ch2 = make_classification(n_samples=200, n_features=2, n_informative=2,
                                   n_redundant=0, n_clusters_per_class=1,
                                   class_sep=1.6, random_state=7)
X_ch2 = StandardScaler().fit_transform(X_ch2)

base_svm = SVC(kernel="linear", C=1.0).fit(X_ch2, y_ch2)
base_lr = LogisticRegression(max_iter=2000).fit(X_ch2, y_ch2)

# Distance from the boundary tells us which points are near and far.
distances = np.abs(base_svm.decision_function(X_ch2))
far_idx = int(np.argmax(distances))               # deepest inside its own side
near_idx = int(np.argmin(distances))              # right on the boundary

def angle_between(w1, w2):
    """Angle in degrees between two boundary normals - how much the boundary rotated."""
    cos = np.dot(w1, w2) / (np.linalg.norm(w1) * np.linalg.norm(w2))
    return np.degrees(np.arccos(np.clip(cos, -1, 1)))


print(f"{'flipped point':<34} {'SVM rotation':>14} {'logistic rotation':>19}")
print("-" * 72)
for label, idx in [("a FAR point (deep in its class)", far_idx),
                   ("a NEAR point (on the boundary)", near_idx)]:
    y_flip = y_ch2.copy()
    y_flip[idx] = 1 - y_flip[idx]
    svm_f = SVC(kernel="linear", C=1.0).fit(X_ch2, y_flip)
    lr_f = LogisticRegression(max_iter=2000).fit(X_ch2, y_flip)
    print(f"{label:<34} {angle_between(base_svm.coef_[0], svm_f.coef_[0]):>13.2f}o "
          f"{angle_between(base_lr.coef_[0], lr_f.coef_[0]):>18.2f}o")

print(f"\nwas the far point a support vector before flipping?  "
      f"{far_idx in base_svm.support_}")
print(f"was the near point a support vector before flipping? "
      f"{near_idx in base_svm.support_}")
print()
print("Flipping a FAR point barely moves the SVM - it was not a support vector, so its")
print("alpha was 0 and it had no vote. Flipping a NEAR point moves it much more, because")
print("that point is on the margin and defines the corridor.")
print()
print("Logistic regression responds to BOTH, because log-loss never reaches zero - every")
print("point always contributes something (Q1).")
print()
print("The lesson cuts both ways. The SVM's insensitivity to far-away points is a genuine")
print("robustness advantage. But it means the model is determined entirely by the hardest,")
print("most ambiguous rows - which are exactly the ones most likely to be mislabelled.")

In [ ]:
# ---------------------------------------------------------------------------
# CHALLENGE 3
# Find the crossover point where a kernel SVM stops being worth it.
#
# For growing n, measure BOTH the fit time and the CV accuracy of SVC(rbf)
# against HistGradientBoostingClassifier. Identify where boosting overtakes on
# accuracy, and where the SVM becomes too slow to cross-validate.
# ---------------------------------------------------------------------------

# YOUR CODE HERE

In [ ]:
# --- Challenge 3: one solution ------------------------------------------
print(f"{'n':>8} {'SVC time':>10} {'SVC acc':>10} {'GB time':>10} {'GB acc':>10} {'winner':>10}")
print("-" * 64)

rows = []
for n in [500, 2000, 8000, 20000]:
    Xn, yn = make_classification(n_samples=n, n_features=25, n_informative=10,
                                 n_redundant=5, flip_y=0.05, random_state=RANDOM_STATE)
    Xn = StandardScaler().fit_transform(Xn)
    Xn_tr, Xn_te, yn_tr, yn_te = train_test_split(Xn, yn, test_size=0.25, stratify=yn,
                                                  random_state=RANDOM_STATE)

    t0 = time.time(); svm_m = SVC().fit(Xn_tr, yn_tr); t_svm = time.time() - t0
    t0 = time.time()
    gb_m = HistGradientBoostingClassifier(random_state=RANDOM_STATE).fit(Xn_tr, yn_tr)
    t_gb = time.time() - t0

    a_svm, a_gb = svm_m.score(Xn_te, yn_te), gb_m.score(Xn_te, yn_te)
    winner = "SVM" if a_svm > a_gb else "boosting"
    rows.append((n, t_svm, a_svm, t_gb, a_gb))
    print(f"{n:>8} {t_svm:>10.3f} {a_svm:>10.4f} {t_gb:>10.3f} {a_gb:>10.4f} {winner:>10}")

print()
svm_growth = rows[-1][1] / rows[0][1]
gb_growth = rows[-1][3] / rows[0][3]
print(f"40x more data cost the SVM {svm_growth:.0f}x more time, boosting {gb_growth:.1f}x.")
print()
print("Two crossovers to notice, and they are different:")
print("  ACCURACY - boosting tends to pull ahead as n grows, because it has the capacity")
print("             to use the extra data and the SVM's smooth RBF boundary saturates.")
print("  COST     - the SVM's time grows super-linearly while boosting's is near-linear.")
print()
print("Remember you must multiply the SVM column by the size of your C x gamma grid, since")
print("both need tuning (Q4). A 20-point grid with 5-fold CV is 100 fits. At the largest n")
print(f"here that is roughly {rows[-1][1] * 100 / 60:.0f} minutes for ONE search.")
print()
print("That multiplication, not the single-fit time, is what actually rules kernel SVMs out.")

---
# Part 5 - Five datasets to practise on

Chosen to cover the regimes where SVMs win and where they lose.

| # | Dataset | Rows × cols | The skill it forces | Difficulty |
|---|---|---|---|---|
| 1 | **Breast cancer** | 569 × 30 | Scaling, the C×gamma grid | ★☆☆☆☆ |
| 2 | **Digits** | 1,797 × 64 | Kernels compared, PCA, multiclass | ★★☆☆☆ |
| 3 | **20 Newsgroups** | ~11k × 100k+ | **Where linear SVMs still win** — sparse text | ★★★☆☆ |
| 4 | **Adult / census** | 48,842 × 14 | Where SVMs lose — scale, categoricals, mixed types | ★★★★☆ |
| 5 | **Gene expression** | ~200 × 20,000 | **p ≫ n**, the SVM's home ground | ★★★★★ |

In [ ]:
from sklearn.datasets import fetch_openml, fetch_20newsgroups_vectorized

CATALOGUE = [
    ("Breast cancer", lambda: load_breast_cancer(as_frame=True)),
    ("Adult (census)", lambda: fetch_openml(name="adult", version=2, as_frame=True)),
]

print(f"{'dataset':<18} {'rows':>7} {'cols':>6} {'NaN':>7}  target")
print("-" * 56)
for label, loader in CATALOGUE:
    try:
        b = loader()
        print(f"{label:<18} {b.data.shape[0]:>7} {b.data.shape[1]:>6} "
              f"{int(b.data.isna().sum().sum()):>7}  {b.target.name}")
    except Exception as exc:
        print(f"{label:<18} unavailable - {type(exc).__name__}: {str(exc)[:30]}")

print(f"{'Digits':<18} {X_dig.shape[0]:>7} {X_dig.shape[1]:>6} {0:>7}  digit 0-9  (bundled)")
print()
print("20 Newsgroups and gene-expression data are larger downloads - loaders are given in")
print("the briefs below rather than fetched here.")

### 1. Breast cancer — the gentle start

```python
from sklearn.datasets import load_breast_cancer
X, y = load_breast_cancer(return_X_y=True)
```

1. Fit `SVC()` with and without a scaler. Quantify the gap (§1.7 gets ~0.07).
2. Grid-search `C` × `gamma` on log scales. Plot the grid as a heatmap — can you see the
   diagonal ridge from §1.6?
3. Compare `kernel="linear"` against `"rbf"`. On 30 well-behaved features, is the extra
   flexibility earning anything?
4. How many support vectors does the best model keep? What does that fraction tell you?
5. Compare against logistic regression on accuracy **and** on Brier score. Which would you
   ship for a *screening* tool, and why (NB-02 Part 3)?

---

### 2. Digits — kernels head to head

Already used in Part 2. Go further:

1. Sweep `PCA(n_components=...)` from 5 to 64 in front of the SVM. Plot accuracy and fit time.
   Why does dimensionality reduction sometimes *help* a kernel method?
2. Tune the polynomial kernel properly — `degree`, `gamma` **and** `coef0`. Can you get it
   competitive with RBF? (Part 2 shows it losing badly at defaults.)
3. `SVC` uses one-vs-one for the 10 classes. Compare against
   `OneVsRestClassifier(SVC())` on accuracy and fit time (Q11).
4. Which digit pairs does the confusion matrix confuse? Fit a binary SVM on just that pair and
   inspect the support vectors as images.

---

### 3. 20 Newsgroups — where linear SVMs still win

```python
from sklearn.datasets import fetch_20newsgroups_vectorized
data = fetch_20newsgroups_vectorized(subset="train", remove=("headers", "footers", "quotes"))
```

Over 100,000 sparse TF-IDF features. **This is the regime linear SVMs own.**

1. `LinearSVC` vs `LogisticRegression` vs `HistGradientBoostingClassifier`. The boosting model
   will do badly — explain why using NB-04 Q9 (random feature subsets on wide sparse data).
2. Do **not** try `SVC(kernel="rbf")` on the full data without thinking about Q6 first.
   Estimate the fit time from §3.1's scaling exponent before you run it.
3. Tune `C` for `LinearSVC`. On text, the optimum is often surprisingly small — why would
   heavy regularisation help with 100k features?
4. Compare `LinearSVC` against `SGDClassifier(loss="hinge")`. Same loss, different solver —
   how close do they get, and how much faster is the second?

---

### 4. Adult / census — where SVMs lose

```python
bunch = fetch_openml(name="adult", version=2, as_frame=True)
```

48k rows, 8 categorical columns, missing values. Deliberately the wrong shape for an SVM.

1. Build the pipeline: impute → one-hot → **scale** → `SVC`. Note how much more preprocessing
   this needs than the tree pipeline in NB-04.
2. Time a single fit. Now estimate a 5-fold search over a 16-point `C`×`gamma` grid. Is that
   feasible on your machine?
3. Use `Nystroem` + `LinearSVC` instead (§3.1). How much accuracy do you recover, at what cost?
4. Compare against `HistGradientBoostingClassifier` with native categorical support. Report
   both accuracy and total wall-clock including preprocessing — that is the honest comparison.

---

### 5. Gene expression — p ≫ n, the SVM's home ground

```python
# e.g. OpenML 'leukemia' (72 x 7129) or any high-dimensional biology dataset
bunch = fetch_openml(name="leukemia", version=1, as_frame=True)
```

~72 samples, ~7,000 features. More features than samples by two orders of magnitude.

1. Fit `SVC(kernel="linear")`. Note that the kernel matrix is only 72×72 — the SVM barely
   notices the 7,000 features. Why is this the regime it was designed for?
2. Try a random forest and gradient boosting. They will likely struggle — connect it to
   NB-04 Q9: with `max_features="sqrt"` ≈ 84 of 7,000 columns, how often does a tree even see
   an informative one?
3. With n=72, a single train/test split is nearly meaningless. Use repeated stratified CV and
   report the **spread**, not just the mean.
4. Try `LinearSVC(penalty="l1", dual=False)` for sparse feature selection. How many genes
   survive, and are they stable across CV folds? (Compare with NB-04 Part 3's warning about
   correlated features — genes are heavily co-expressed.)

---
# Part 6 - Reading the literature

SVMs have one of the cleanest literatures in ML — the foundational papers are short, precise
and still readable.

## Start here

**1. [Support-Vector Networks](https://link.springer.com/article/10.1007/BF00994018)** —
Corinna Cortes & Vladimir Vapnik, *Machine Learning* 20(3):273-297, 1995.
> **The paper.** Introduces the soft-margin SVM — the slack variables and `C` of §1.3 — and is
> the reference everyone cites. Sections 1–3 are the whole of Part 1 of this notebook, written
> more precisely and in about ten pages. Genuinely worth reading in full.

**2. [A Training Algorithm for Optimal Margin Classifiers](https://dl.acm.org/doi/10.1145/130385.130401)** —
Boser, Guyon & Vapnik, COLT 1992.
> The **kernel trick** paper — three years earlier, and the more surprising idea. This is where
> "replace the inner product and you get a non-linear classifier for free" first appears.

**3. [A Practical Guide to Support Vector Classification](https://www.csie.ntu.edu.tw/~cjlin/papers/guide/guide.pdf)** —
Hsu, Chang & Lin, 2003. **Free.**
> Not a research paper — a **recipe** from the authors of libsvm (which sklearn wraps). Scale
> your features, use RBF, grid-search `C` and `gamma` on log scales, in that order. Everything
> in §1.6 and §1.7 of this notebook is in this 16-page document, and it is the single most
> practically useful thing on this list.

## The paper behind each section

| Section | Source | Free? |
|---|---|---|
| 1.1–1.2 — margins, support vectors, the dual | **Cortes & Vapnik**, *Support-Vector Networks*, **1995** | 🔍 |
| 1.3 — soft margin and `C` | **Cortes & Vapnik**, **1995** — the slack variables are the paper's main contribution | 🔍 |
| 1.4 — hinge loss, and the link to regularised risk | Hastie, Tibshirani & Friedman, *ESL* ch. 12 — [free PDF](https://hastie.su.domains/ElemStatLearn/) | ✅ |
| 1.5 — **the kernel trick** | **Boser, Guyon & Vapnik**, COLT **1992** | 🔍 |
| 1.5 — Mercer's condition, kernel theory | **Schölkopf & Smola**, *Learning with Kernels*, MIT Press, **2002** (book) | 🔍 |
| 1.6–1.7 — the practical recipe | **Hsu, Chang & Lin**, **2003** — [pdf](https://www.csie.ntu.edu.tw/~cjlin/papers/guide/guide.pdf) | ✅ |
| 3.1 — the solver sklearn actually uses | **Chang & Lin**, *LIBSVM: A Library for Support Vector Machines*, ACM TIST 2(3), **2011** — [pdf](https://www.csie.ntu.edu.tw/~cjlin/papers/libsvm.pdf) | ✅ |
| 3.1 — SMO, which made SVMs trainable | **Platt**, *Sequential Minimal Optimization*, MSR-TR-98-14, **1998** | 🔍 |
| 3.1 — kernel approximation (Nystroem / random features) | **Rahimi & Recht**, *Random Features for Large-Scale Kernel Machines*, NeurIPS **2007** — [pdf](https://papers.nips.cc/paper_files/paper/2007/hash/013a006f03dbc5392effeb8f18fda755-Abstract.html) | ✅ |
| 3.2 — Platt scaling | **Platt**, *Probabilistic Outputs for Support Vector Machines*, **1999** | 🔍 |
| 3.3 — SVR | **Drucker et al.**, *Support Vector Regression Machines*, NeurIPS **1996** | 🔍 |
| Q10 — why they faded on tabular data | **Grinsztajn, Oyallon & Varoquaux**, NeurIPS **2022** — [arXiv](https://arxiv.org/abs/2207.08815) | ✅ |

**Legend:** ✅ free at the link · 🔍 search the exact title on
[Google Scholar](https://scholar.google.com) — Vapnik's papers are widely mirrored

### If you read only one

**Hsu, Chang & Lin (2003).** It is free, it is sixteen pages, and it will make you better at
actually using an SVM tomorrow. Read Cortes & Vapnik afterwards for *why* the recipe works.

---
# Appendix

| Symptom | Cause | Fix |
|---|---|---|
| Terrible accuracy, no obvious reason | **Unscaled features** (§1.7) — by far the most common | `StandardScaler` inside the Pipeline |
| Fitting never finishes | $O(n^2)$–$O(n^3)$ scaling (§3.1) | `LinearSVC`, `SGDClassifier`, `Nystroem`, or boosting |
| Nearly every point is a support vector | `C` too small or `gamma` too large | Tune both together on log scales (§1.6) |
| Training accuracy ~1.0, test poor | `gamma` far too large — islands around points | Lower `gamma`, lower `C` |
| Almost a straight line on curved data | `gamma` too small | Raise `gamma`, or check scaling first |
| `FutureWarning: probability parameter was deprecated` | `SVC(probability=True)` removed in sklearn 1.11 (§3.2) | `CalibratedClassifierCV(SVC(), ensemble=False)` |
| Getting probabilities is ~6x slower | Calibration fits the SVM once per CV fold (§3.2) | Only calibrate if you truly need probabilities; otherwise use `decision_function` |
| `ConvergenceWarning` from `LinearSVC` | Not converged | Raise `max_iter`; scale features; try `dual="auto"` |
| Memory error on a big dataset | The $n\times n$ kernel matrix | See "fitting never finishes" |
| Results shift between runs | Calibration uses internal CV | Set `random_state` on the CV splitter |
| Poor results on imbalanced data | Margin ignores class frequency | `class_weight="balanced"`, then tune the threshold |

## Checklist for shipping an SVM

Everything in the Foundations checklist, plus:

- [ ] Are features **scaled**, with the scaler **inside** the Pipeline?
- [ ] Were `C` and `gamma` tuned **together**, on log scales?
- [ ] What fraction of the training data ended up as support vectors — and does that fraction
      suggest over- or under-fitting?
- [ ] Is $n$ small enough that this will still train when the data grows?
- [ ] If I need probabilities, am I using `CalibratedClassifierCV` deliberately — and am I
      off the deprecated `probability=True` (removed in sklearn 1.11)?
- [ ] For imbalanced data: `class_weight` set, and threshold chosen from costs (NB-02 Part 3)?
- [ ] Have I compared against a linear model **and** a tree ensemble, so I know what the kernel
      bought?
- [ ] Does the production preprocessing reproduce the training scaling *exactly*?

## Where to go next

| Notebook | Why it follows |
|---|---|
| [`knn_zero_to_hero.ipynb`](knn_zero_to_hero.ipynb) | The other distance-based model. Same scaling dependency, same curse of dimensionality, very different treatment of what "nearby" means. |
| `pca_zero_to_hero.ipynb` | Part 2 used PCA in front of the SVM and it helped. PCA is also kernelisable — the same trick, applied to a different algorithm. |
| `anomaly_detection_zero_to_hero.ipynb` | `OneClassSVM` in full, alongside Isolation Forest and LOF. |

See [`ZERO_TO_HERO_PLAN.md`](../ZERO_TO_HERO_PLAN.md) for the full roster and status.